### Cluster 4 (658 companies, 115 bankrupt)
### Strategy: Weighted Stacking

In [1]:
# Tanishq_4.ipynb


# ==========================================
# 1. Imports
# ==========================================
import pandas as pd
import numpy as np
import joblib
import os
import sys

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold

In [2]:
# Configuration
CLUSTER_ID = 4
INPUT_FILE = f"cluster_{CLUSTER_ID}.csv"
FEATURE_FILE = "top_features_for_clustering.joblib"
OUTPUT_MODEL = f"cluster{CLUSTER_ID}_stacking.joblib"
RANDOM_STATE = 0

# ==========================================
# 2. Data Loading
# ==========================================

print(f"Loading data for Cluster {CLUSTER_ID}...")
df = pd.read_csv(INPUT_FILE)
print(f"Shape: {df.shape}")
print(f"Class Distribution:\n{df['Bankrupt?'].value_counts()}")

# Load features
features_to_use = joblib.load(FEATURE_FILE)
print(f"Loaded {len(features_to_use)} features.")

X = df[features_to_use]
y = df["Bankrupt?"]

Loading data for Cluster 4...
Shape: (658, 98)
Class Distribution:
Bankrupt?
0    543
1    115
Name: count, dtype: int64
Loaded 40 features.


In [3]:
# ==========================================
# 3. Define Base Models
# ==========================================
# We use 'balanced' class weights to handle the 17% imbalance naturally.

# Base 1: Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Base 2: Gradient Boosting (Standard, robust)
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=RANDOM_STATE
)

# Base 3: KNN (Effective for this cluster size ~600 rows)
knn = KNeighborsClassifier(n_neighbors=5)

estimators = [
    ('rf', rf),
    ('gb', gb),
    ('knn', knn)
]

In [4]:
# ==========================================
# 4. Define & Train Stacking Model
# ==========================================
# Standard 5-fold CV is safe here because we have 115 positives (enough for every fold)
clf_stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced'), # Weighting the meta-learner too
    cv=5, 
    n_jobs=-1
)

print("\nTraining Stacking Model...")
clf_stack.fit(X, y)
print("Training Complete.")


Training Stacking Model...
Training Complete.


In [5]:
# ==========================================
# 5. Evaluation & Compliance (Table 3)
# ==========================================
def get_metrics(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    FF, FT, TF, TT = cm.ravel()
    
    # Equation 1 Accuracy: TT / (TF + TT)
    acc_eq1 = TT / (TF + TT) if (TF + TT) > 0 else 0
    
    print(f"{name}")
    print(f"Confusion Matrix: [[{FF}, {FT}], [{TF}, {TT}]]")
    print(f"Eq(1) Accuracy: {acc_eq1:.4f}")
    return FF, FT, TF, TT, acc_eq1

print(f"RESULTS FOR TABLE 3 (Cluster {CLUSTER_ID})")

# A. Base Model Performance
# We use the fitted base models inside the stack
print("\nBase Model Performance (on full train set):")
for name, est in clf_stack.named_estimators_.items():
    y_pred_base = est.predict(X)
    get_metrics(y, y_pred_base, f"Base: {name}")

# B. Stacking Model Performance
print("\nStacking Model Performance (Final):")
y_pred_stack = clf_stack.predict(X)
FF, FT, TF, TT, acc = get_metrics(y, y_pred_stack, "STACKING MODEL")

# ==========================================
# 6. Save Model Package
# ==========================================
model_package = {
    "cluster_id": CLUSTER_ID,
    "features": features_to_use,
    "model": clf_stack,
    "stats": {"TT": TT, "TF": TF, "N_features": len(features_to_use)}
}

joblib.dump(model_package, OUTPUT_MODEL)
print(f"\nModel saved to {OUTPUT_MODEL}")
print(f"Stats for Table 3 -> TT: {TT}, TF: {TF}")

RESULTS FOR TABLE 3 (Cluster 4)

Base Model Performance (on full train set):
Base: rf
Confusion Matrix: [[543, 0], [0, 115]]
Eq(1) Accuracy: 1.0000
Base: gb
Confusion Matrix: [[543, 0], [10, 105]]
Eq(1) Accuracy: 0.9130
Base: knn
Confusion Matrix: [[532, 11], [88, 27]]
Eq(1) Accuracy: 0.2348

Stacking Model Performance (Final):
STACKING MODEL
Confusion Matrix: [[531, 12], [0, 115]]
Eq(1) Accuracy: 1.0000

Model saved to cluster4_stacking.joblib
Stats for Table 3 -> TT: 115, TF: 0
